# SPECTRA-SR — Colab big run

Run the cells in order. Cells 1–5 are setup and checks; **do not start cell 6 (training) until cell 4 says the backup works and cell 5 has given you a batch size.**

Two things this notebook deliberately does *not* do:

- **It does not mount Google Drive for the dataset.** The data lives on the HuggingFace Hub, and Colab's link to HF is far faster than an upload from a laptop followed by a Drive read. Pulling directly also means the laptop never has to hold or transfer the full dataset.
- **It does not keep checkpoints only on the Colab disk.** That disk disappears when the session ends. Cell 4 verifies the off-machine backup *before* any compute is spent, because a backup that turns out to be misconfigured is discovered at exactly the worst moment otherwise.

## 1. GPU check

Confirm what was actually allocated — Colab hands out different GPUs, and the batch size in cell 5 depends on which one you got.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Code

Clone the repo and install what Colab does not already have. `rasterio` is the only heavy addition; torch is preinstalled.

Repo is public for this run, so a plain clone works -- no token needed. If it goes private again later, this cell needs a GitHub token added to Colab's Secrets panel (same pattern as `HF_TOKEN` below) and the clone URL updated to use it.


In [ ]:
import os
REPO_URL = 'https://github.com/ManasMehta1110/Spectra-SR.git'
if not os.path.exists('/content/spectra-sr'):
    !git clone $REPO_URL /content/spectra-sr
%cd /content/spectra-sr
!pip install -q rasterio huggingface_hub
os.environ['PYTHONPATH'] = '/content/spectra-sr/src'
!python -c "import sys; sys.path.insert(0,'src'); import spectra_sr, rasterio; print('imports ok')"


## 3. Data

Two releases, and the difference is **not** cosmetic:

| | v1 cross-sensor | **v2 cross-sensor** | v1/v2 synthetic |
|---|---|---|---|
| pairs | 2,851 | **8,000** | 17,657 / ~61k |
| size | 2.2 GB | **9.7 GB** | 180 GB / ~140 GB |
| Sentinel-2 | real | **real** | none - LR is simulated |
| HR dtype | uint8 NAIP (/255) | **uint16 reflectance (/10000)** | uint8 NAIP |
| radiometric calibration | required | **already harmonized** | n/a |

**v2 is the training set**: 2.8x more real pairs, every one within a day, cloud-free, and its HR is already on the Sentinel-2 radiometric scale (measured per-band HR/LR ratios 1.000/0.999/0.999/1.000).

**But v2 is also an easier task, so its numbers are not comparable to v1's.** Its HR was harmonized *using the real Sentinel-2 as reference*, which pulls the target toward the input: measured correlation between avg-pooled HR and LR is 0.995 on v2 versus 0.850 on v1. Model-vs-bicubic comparisons stay valid within a release (both face the same target); a PSNR jump from switching releases would be an easier target, not a better model.

So: **train on v2, and evaluate on v1 as well** - held-out v1 is the harder, more realistic pairing and the one every earlier number was measured on.


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import zipfile, glob, os

DATA = '/content/data'

# v2 cross-sensor -- the training set. Ships as a legacy TACO container.
hf_hub_download('tacofoundation/SEN2NAIPv2', filename='sen2naipv2-crosssensor.taco',
                repo_type='dataset', local_dir=f'{DATA}/sen2naipv2')

# tacoreader<1.0 pins pandas 3.x / numpy 2.5.x, which conflict with the training stack. Install
# it to a separate directory and put that FIRST on sys.path only for the extraction step, so the
# training environment never imports it.
!pip install -q --target /content/tacolegacy 'tacoreader<1.0'
!PYTHONPATH=/content/tacolegacy python scripts/extract_sen2naipv2.py --taco $DATA/sen2naipv2/sen2naipv2-crosssensor.taco --out $DATA/sen2naipv2/cross-sensor

# v1 cross-sensor -- kept as the harder held-out evaluation set, not for training.
snapshot_download('isp-uv-es/SEN2NAIP', repo_type='dataset',
                  allow_patterns=['cross-sensor/cross-sensor.zip'],
                  local_dir=f'{DATA}/sen2naip', max_workers=8)
zp = f'{DATA}/sen2naip/cross-sensor/cross-sensor.zip'
if os.path.exists(zp):
    with zipfile.ZipFile(zp) as z:
        z.extractall(os.path.dirname(zp))
    os.remove(zp)

!du -sh /content/data/*


## 4. Backup — verify BEFORE training

Set `HF_TOKEN` to a **write**-scoped token. A read-scoped token authenticates perfectly and only fails at the first real upload, which on a long run means finding out hours in. The check below round-trips an actual write, so it catches that now.

Use Colab's Secrets panel (🔑 in the sidebar) rather than pasting the token into a cell — cell contents get saved into the notebook file.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['SPECTRA_SR_HF_REPO'] = 'ManasMehta/spectra-sr-checkpoints'  # your repo

!python scripts/hf_checkpoint.py check

## 5. Memory probe — pick the batch size from measurement

`FULL` has never been trained. It OOM'd on the 4.3 GB dev laptop, which tells you nothing about a 16 GB T4 or a 40 GB A100. This measures a real training step (forward + backward + optimizer), because backward is what actually peaks — a forward-only probe under-reports by roughly 2–3×.

Take the largest batch that stays under ~85% of the card: validation allocates on top of the training peak, and fragmentation over a long run needs headroom.

In [ ]:
!python scripts/probe_memory.py --configs colab_realistic full --batch-sizes 1 2 4 8 16 32

## 6. Train

Set `BATCH_SIZE` from cell 5. `--hf-backup-every-n-epochs 1` makes every epoch survive the session ending.

`--val-every-n-steps 1000` gives a dense validation curve (~200 points over a long run) instead of one point per epoch, at under 1% overhead — enough resolution to see a divergence while there is still time to kill the run.

In [ ]:
CONFIG      = 'full'
BATCH_SIZE  = 8      # <- from cell 5
EPOCHS      = 60
OUT         = '/content/spectra-sr/checkpoints/big_run'

# --sen2naip-variant v2 selects the 520/130 tile geometry, the /10000 HR scaling and
# calibration-off. Getting that wrong is silent: v1's /255 rule applied to v2 overshoots by ~40x
# and trains on nonsense without raising anything.
!python scripts/train_pretrain.py --config $CONFIG --batch-size $BATCH_SIZE --epochs $EPOCHS --out $OUT --data-source sen2naip --sen2naip-variant v2 --sen2naip-dir /content/data/sen2naipv2/cross-sensor --res-scale 0.2 --lr 2e-4 --grad-clip-norm 5.0 --sen2naip-train-crops 2 --sen2naip-val-crops 1 --val-every-n-steps 1000 --step-val-tiles 64 --hf-backup-every-n-epochs 1 --keep-last-n 3


## 7. If the session dropped

Re-run cells 1–4, then this. `--resume` restores the optimizer moments, the LR-schedule position, the best-so-far val loss and the global step — reloading weights alone would silently restart Adam and the cosine schedule from zero, which is not the same run continued.

**`--epochs` must match the original run.** The cosine schedule's period is the epoch count, so resuming into a different-length schedule produces an LR curve matching neither run. The script refuses rather than letting that happen quietly.

In [ ]:
!python scripts/hf_checkpoint.py pull /content/spectra-sr/checkpoints/restored
!ls -la /content/spectra-sr/checkpoints/restored/big_run/

# then re-run cell 6 with:
#   --resume /content/spectra-sr/checkpoints/restored/big_run/checkpoint_epoch<N>.pt
# and the SAME --epochs value as the original run.